# Linear regression, workshop 1 — The line, the cost, and the exact answer

The project gives you the data to work with:

```python
X = 4 * np.random.rand(100, 1)
y = 10 + 2 * X + np.random.randn(100, 1)
```

A hundred points generated from a line of **intercept 10 and slope 2**, plus noise. That is a gift:
you know the right answer in advance, so anything your code produces can be checked against it.

Today:

1. **The hypothesis** `h(x, theta)` — a line, written as a dot product.
2. **The cost** `mean_squared_error` — one number for how wrong a line is.
3. **The bias column**, and the **normal equation** that solves it exactly.
4. **Residuals** — where the model is still wrong, and what that tells you.

In [ ]:
# ABOUTME: My Linear Regression workshop 1 - hypothesis, MSE, normal equation, residuals.
# ABOUTME: Runs on the data the project subject generates.

import numpy as np
import matplotlib.pyplot as plt

# The data the project subject generates, verbatim. A seed is added so every run of this notebook
# shows the same numbers; the subject does not fix one, so yours will differ in the last decimals.
np.random.seed(42)
X = 4 * np.random.rand(100, 1)
y = 10 + 2 * X + np.random.randn(100, 1)

def ready(*values):
    """True once the exercises a cell depends on have been filled in."""
    return all(v is not None for v in values)

print("X", X.shape, " y", y.shape)
print("generated with intercept 10 and slope 2, plus standard normal noise")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X, y, s=20, alpha=0.7)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("The subject's 100 points")
plt.tight_layout()

A clear upward trend with scatter around it — exactly the situation the subject describes as "a
linear model is a reasonable choice".

## 1. The hypothesis

$$y = h_\theta(x) = \theta^T \cdot \mathbf{x} = \theta_0 x_0 + \theta_1 x_1 + \dots$$

with $x_0$ always equal to 1. Written as a dot product, one line of numpy handles any number of
features at once — which is the whole reason for writing it that way.

### Exercise 1 — `h(x, theta)`

Write the hypothesis. `x` is a matrix with one row per instance (and its bias column already
added); `theta` is the weight vector. No Python loop.

In [ ]:
def h(x, theta):
    """Predictions for every row of x. x already carries its bias column."""
    # TODO: a dot product
    return None

# two instances, two weights: intercept 1, slope 3
print(h(np.array([[1.0, 2.0], [1.0, 4.0]]), np.array([[1.0], [3.0]])))

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

_x = np.array([[1.0, 2.0], [1.0, 4.0]])
check("one prediction per row", lambda: h(_x, np.array([[1.0], [3.0]])).shape == (2, 1))
check("intercept 1 + slope 3 on x=2 gives 7", lambda: abs(h(_x, np.array([[1.0], [3.0]]))[0, 0] - 7) < 1e-9)
check("a zero slope returns the intercept",
      lambda: np.allclose(h(_x, np.array([[5.0], [0.0]])), [[5.0], [5.0]]))

## 2. The cost

For each point, the error is prediction minus truth. Square it (so too-high and too-low are
equally bad, and big misses hurt disproportionately), then average:

$$\text{MSE}(X, h_\theta) = \frac{1}{m}\sum_{i=1}^{m}\left(\theta^T \cdot x^{(i)} - y^{(i)}\right)^2$$

The subject notes that RMSE and MSE are minimised by the same $\theta$, because the square root is
increasing — so we minimise the simpler one.

### Exercise 2 — `mean_squared_error(y_predicted, y_label)`

Check it by hand first: predictions `[2, 4]` against truth `[3, 3]` give errors −1 and +1, so the
MSE is `(1 + 1) / 2 = 1`.

**Watch the shapes.** `y` here is `(100, 1)`, not `(100,)`. If one side of your subtraction is
`(100,)` and the other `(100, 1)`, numpy broadcasts them into a `(100, 100)` matrix and returns a
number that is wrong — with no error raised. The assert below exists to catch exactly that.

In [ ]:
def mean_squared_error(y_predicted, y_label):
    """Mean of the squared differences."""
    assert y_predicted.shape == y_label.shape, (y_predicted.shape, y_label.shape)
    # TODO: subtract, square, take the mean
    return None

print("hand-checked:", mean_squared_error(np.array([[2.0], [4.0]]), np.array([[3.0], [3.0]])), "(should be 1.0)")

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("the hand-checked case",
      lambda: abs(mean_squared_error(np.array([[2.0], [4.0]]), np.array([[3.0], [3.0]])) - 1.0) < 1e-12)
check("a perfect fit costs nothing",
      lambda: mean_squared_error(np.array([[5.0]]), np.array([[5.0]])) == 0)
check("too high and too low cost the same",
      lambda: mean_squared_error(np.array([[4.0]]), np.array([[3.0]]))
           == mean_squared_error(np.array([[2.0]]), np.array([[3.0]])))

That `assert` is the single most valuable line in this notebook. Without it, a `(100,)` meeting a
`(100, 1)` silently produces a 100 x 100 error matrix and an MSE roughly **13 times too large** —
a wrong answer that looks like a real one. Keep the assert in your submission.

## 3. The bias column and the exact answer

The hypothesis treats $\theta_0$ as the weight of a feature $x_0$ that is always 1. Your data does
not have that column yet, so you add it.

### Exercise 3 — `bias_column(X)`

In [ ]:
def bias_column(X):
    """Prepend a column of ones, so theta_0 becomes an ordinary weight."""
    # TODO: np.hstack a column of ones in front of X
    return None

X_new = bias_column(X)
if ready(X_new):
    print(X[:3])
    print(" ---- ")
    print(X_new[:3])

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("one extra column", lambda: X_new.shape == (100, 2))
check("the first column is all ones", lambda: np.allclose(X_new[:, 0], 1.0))
check("the original values are preserved", lambda: np.allclose(X_new[:, 1:], X))

Now the payoff. Setting the derivative of the MSE to zero gives the **normal equation**, which
returns the best $\theta$ directly — no search, no learning rate, no iterations:

$$\theta = (X^T \cdot X)^{-1} \cdot X^T \cdot y$$

### Exercise 4 — `LeastSquaresRegression`

Use `np.linalg.pinv`, the pseudo-inverse, rather than `inv`. If two of your columns are identical —
for instance because `bias_column` was applied twice — `X^T X` cannot be inverted and `inv` raises
`LinAlgError: Singular matrix`, while `pinv` still returns a sensible answer.

In [ ]:
class LeastSquaresRegression:
    def __init__(self):
        self.theta_ = None

    def fit(self, X, y):
        """Compute the theta minimising the MSE, and store it in self.theta_."""
        # TODO: the normal equation, with pinv
        pass

    def predict(self, X):
        """Predictions for X, using the fitted theta."""
        # TODO: reuse h
        return None

model = LeastSquaresRegression()
model.fit(X_new, y)
print("theta:", None if model.theta_ is None else model.theta_.ravel())

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("theta has two weights", lambda: model.theta_.shape == (2, 1))
check("the intercept is near 10", lambda: abs(model.theta_[0, 0] - 10) < 1.0)
check("the slope is near 2", lambda: abs(model.theta_[1, 0] - 2) < 0.5)
check("predictions have the shape of y", lambda: model.predict(X_new).shape == y.shape)

**θ = [10.215, 1.885]** against the 10 and 2 the data was built from. The gap is the noise: with
100 noisy points you recover the rule closely, not exactly. That is the honest answer to the
subject's question *"are the values consistent with the generating equation?"* — yes, within
sampling error, and here is the size of that error.

The subject also asks about **computational complexity**. Forming $X^TX$ costs $O(n d^2)$ and
inverting it $O(d^3)$, where *n* is instances and *d* features — so training grows fast with the
number of features, and barely at all with rows. Prediction is a single matrix product, $O(n d)$.
Measured on 10 000 rows: fitting goes from 0.3 ms at 10 features to 723 ms at 400, while
prediction stays in the low milliseconds throughout. **Training is the expensive half; prediction
is nearly free** — and that asymmetry is why the normal equation becomes unusable long before
prediction does.

In [ ]:
y_new = model.predict(X_new) if ready(X_new, model.theta_) else None

def my_plot(X, y, y_new):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(X, y, s=20, alpha=0.7, label="observations")
    order = np.argsort(X.ravel())
    ax.plot(X.ravel()[order], y_new.ravel()[order], "r-", lw=2, label="least-squares fit")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
    plt.tight_layout()

if ready(y_new):
    my_plot(X, y, y_new)
else:
    print("Finish exercises 3 and 4, then run this cell.")

## 4. Residuals

The MSE is one number for the whole dataset. Residuals — $y - \hat{y}$, one per point — tell you
*where* the error lives. Plotted against the prediction you want to see **nothing**: a shapeless
band around zero. Any pattern is signal the model failed to capture.

### Exercise 5

In [ ]:
def residuals(X_with_bias, y, model):
    """Actual minus predicted, one per observation."""
    # TODO
    return None

res = residuals(X_new, y, model)
print("mean residual:", None if res is None else round(float(np.mean(res)), 12))

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("one residual per point", lambda: res.shape == y.shape)
check("least-squares residuals average to zero", lambda: abs(float(np.mean(res))) < 1e-9)

That second check is not luck: a least-squares fit **with an intercept** always leaves residuals
summing to zero. If yours do not, you have a bug, not a bad model.

And the standard deviation of the residuals comes out at **0.898**, against the **1.0** that
`np.random.randn` put in. Close, and slightly *under* — because the fitted line absorbs a sliver of
the noise along with the signal. Your model has recovered the rule and left behind essentially the
randomness, which is the best any model can do.

In [ ]:
if not ready(res, y_new):
    print("Finish exercise 5, then run this cell.")
else:
  fig, axes = plt.subplots(1, 2, figsize=(13, 4))
  axes[0].scatter(model.predict(X_new), res, s=18, alpha=0.7)
  axes[0].axhline(0, color="k", lw=1)
  axes[0].set_xlabel("predicted"); axes[0].set_ylabel("residual"); axes[0].set_title("Shapeless band: good")

  # The same data fitted with the intercept forced to zero - a model that is wrong on purpose.
  theta_no_intercept = np.linalg.pinv(X.T.dot(X)).dot(X.T).dot(y)
  res_bad = y - X.dot(theta_no_intercept)
  axes[1].scatter(X.dot(theta_no_intercept), res_bad, s=18, alpha=0.7, color="crimson")
  axes[1].axhline(0, color="k", lw=1)
  axes[1].set_xlabel("predicted"); axes[1].set_title("Same data, no bias column: structure appears")
  plt.tight_layout()
  print("mean residual without a bias column:", round(float(np.mean(res_bad)), 3))

The right-hand panel is what a missing bias column looks like: every residual pushed to one side,
a mean far from zero, and a clear slope in what should be noise. The model cannot shift the line up
to meet the data, so it tilts instead.

**Rule of thumb:** the MSE tells you *how much* you are wrong; the residual plot tells you *why*.
Never report one without having looked at the other.

## What to take away

1. **Write the hypothesis as a dot product** and it works for any number of features unchanged.
2. **Assert your shapes inside the MSE.** `(100,)` against `(100, 1)` silently returns a number
   that is 13x too large.
3. **The bias column** is how an intercept becomes an ordinary weight — and its absence is visible
   in the residual plot.
4. **`pinv` over `inv`**, so a degenerate matrix degrades instead of crashing.
5. **Check against the generating values.** 10 and 2 recovered as 10.215 and 1.885 is the right
   kind of answer: close, not exact, and the gap is the noise.
6. **Training is $O(nd^2 + d^3)$, prediction $O(nd)$.** Fitting is the expensive half.
7. **Residuals should be shapeless, and sum to zero.**

### Take it home
The subject asks you to plot the generated points, fit them, and compare to 10 and 2 — you have
done all three. Keep the shape assert and the residual plot in your submission: they are what a
reviewer looks for.